In [1]:
import torch.nn as nn
from torchvision import transforms as T, models
import torch
from AgePredictorCORAL import CoralEfficientNetV2
from AgePredictorRegression import EfficientNetV2Regression
from PIL import Image
import numpy as np
# from deepface import DeepFace
import cv2

In [10]:

class AgePredictor:
    def __init__(self, weights_path = './models/EfficientNetLRegression.pth', device="cpu"):
        self.device = torch.device(device)

        # 1. Crear modelo
        self.model = EfficientNetV2Regression().to(self.device)

        # 2. Cargar pesos
        state = torch.load(weights_path, map_location=self.device)
        self.model.load_state_dict(state['model_state_dict'])
        self.model.eval()

        # 3. Definir transform para inference (val_transform)
        self.transform = T.Compose([
            T.Resize((244, 244)),
            T.ToTensor(),
            T.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225]),
        ])

    
    def load_image(self, image_path):
        
        img_pillow = Image.open(image_path).convert("RGB")
        img_array = np.array(img_pillow)
        return img_array

    def retina_face(self, image_path):
        image_array = self.load_image(image_path)
        
        img_objs = DeepFace.extract_faces(
            img_path = image_array,
            detector_backend = 'retinaface', 
            align = True, 
            expand_percentage = 10,
            enforce_detection = False
        )
        
        area = img_objs[0]['facial_area']
        x, y, w, h = area['x'], area['y'], area['w'], area['h']
        face_crop = image_array[y:y+h, x:x+w]
    
        face_img = Image.fromarray(face_crop)
        return image_array, face_img, (x, y, w, h)
    
    def predict(self, image_path):
        
        
        original_img, img_array, _ = self.retina_face(image_path)
        img = self.transform(img_array).unsqueeze(0).to(self.device)  # (1,3,H,W)

        with torch.no_grad():
            output = self.model(img) 
            pred_age = output.item()

        pred_age = int(pred_age)

        return pred_age, img_array, original_img
    


class CoralAgePredictor:
    def __init__(self, weights_path='./models/EfficientNetLCORAL.pth', device="cpu", max_age=80):
        self.device = torch.device(device)
        self.max_age = max_age

        # 1. Crear modelo CORAL
        self.model = CoralEfficientNetV2(max_age=max_age).to(self.device)

        # 2. Cargar pesos
        state = torch.load(weights_path, map_location=self.device)
        self.model.load_state_dict(state['model_state_dict'])
        self.model.eval()

        # 3. Transform
        self.transform = T.Compose([
            T.Resize((244, 244)),
            T.ToTensor(),
            T.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225]),
        ])

    def load_image(self, image_path):
        img_pillow = Image.open(image_path).convert("RGB")
        return np.array(img_pillow)

    def retina_face(self, image_path):
        image_array = self.load_image(image_path)
        
        img_objs = DeepFace.extract_faces(
            img_path=image_array,
            detector_backend='retinaface',
            align=True,
            expand_percentage=10,
            enforce_detection=False
        )
        
        area = img_objs[0]['facial_area']
        x, y, w, h = area['x'], area['y'], area['w'], area['h']
        face_crop = image_array[y:y+h, x:x+w]
        face_img = Image.fromarray(face_crop)

        return image_array, face_img, (x, y, w, h)

    def predict(self, image_path):
        original_img, face_img, _ = self.retina_face(image_path)

        img = self.transform(face_img).unsqueeze(0).to(self.device)

        with torch.no_grad():
            logits = self.model(img)  # (1, max_age)
            pred_age = self.predict_age(logits).item()  # convierte umbrales → edad

        return int(pred_age), face_img, original_img
    
    def predict_age(self, logits):
        probs = torch.sigmoid(logits)
        return (probs > 0.5).sum(dim=1)



In [11]:
regression_predictor = AgePredictor(device="cpu")
coral_predictor = CoralAgePredictor(device="cpu")